<a href="https://colab.research.google.com/github/Kraven15/Unicafam_Modelos_Analiticos_Indra/blob/main/Sesion_3_integracion_de_datos_Jorge_Andrade_Ejercicios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO

csv_clientes = """id_cliente;nombre;ciudad;segmento
C001;Ana Ruiz;Bogotá;Oro
C002;Luis Peña;Medellín;Plata
C003;Marta Gil;Cali;Oro
C004;Juan Ríos;Bogotá;Bronce
C005;Sofía Mora;Barranquilla;Plata"""

clientes = pd.read_csv(StringIO(csv_clientes), sep=";")


In [ ]:
# Se agrega 3 clientes

# creacion de la columna total
nuevos_clientes = pd.DataFrame({
    "id_cliente": ["C006", "C007", "C008"],
    "nombre": ["Pedro Ruiz", "Tony Stark", "Bruce wayne",  ],
    "ciudad": ["Medellín", "Bogotá", "Barranquilla" ],
    "segmento": [" Bronce", "Plata", "Oro" ]
})

clientes = pd.concat([clientes, nuevos_clientes], ignore_index=True)


In [ ]:
# lista los clientes sin compras.
pedidos = pd.DataFrame({
    "id_pedido": ["P001", "P002", "P003", "P004", "P005"],
    "cliente": ["C001", "C002", "C003", "C001", "C009"],
    "compras": [150000, 200000, 350000, 80000, 120000]
})


m = pd.merge(clientes, pedidos,
             left_on="id_cliente",
             right_on="cliente",
             how="left")

sin_compras = m[m["cliente"].isna()]

print(sin_compras[["id_cliente", "nombre"]])

In [33]:
# cuenta cada categoría
m = pd.merge(clientes, pedidos,
             left_on="id_cliente",
             right_on="cliente",
             how="outer",
             indicator=True)

print(m["merge"].value_counts())

_merge
left_only     21
both           4
right_only     1
Name: count, dtype: int64


In [49]:
marketing = m[m["_merge"] == "left_only"]

print(marketing[["id_cliente", "nombre", "ciudad", "segmento"  ]])


clientes = clientes.drop_duplicates(subset="id_cliente")

   id_cliente       nombre        ciudad segmento
4        C004    Juan Ríos        Bogotá   Bronce
5        C005   Sofía Mora  Barranquilla    Plata
6        C006       Pedro       Medellín   Bronce
7        C006       Pedro       Medellín   Bronce
8        C006       Pedro       Medellín   Bronce
9        C006       Pedro       Medellín   Bronce
10       C006       Pedro       Medellín   Bronce
11       C006       Pedro       Medellín   Bronce
12       C006       Pedro       Medellín   Bronce
13       C006   Pedro Ruiz      Medellín   Bronce
14       C006   Pedro Ruiz      Medellín   Bronce
15       C006   Pedro Ruiz      Medellín   Bronce
16       C006   Pedro Ruiz      Medellín   Bronce
17       C007   Tony Stark        Bogotá    Plata
18       C007   Tony Stark        Bogotá    Plata
19       C007   Tony Stark        Bogotá    Plata
20       C007   Tony Stark        Bogotá    Plata
21       C008  Bruce wayne  Barranquilla      Oro
22       C008  Bruce wayne  Barranquilla      Oro


In [48]:
inexistentes = m[m["_merge"] == "right_only"]

print(inexistentes[["id_pedido", "cliente", "compras"]])

   id_pedido cliente   compras
25      P005    C009  120000.0


In [44]:
import pandas as pd

# Crear los datos de cada mes
enero = pd.DataFrame({
    "sucursal": ["Bogota", "Medellin", "Cali"],
    "mes": ["Ene", "Ene", "Ene"],
    "ventas": [1000000, 2000000, 1500000]
})

febrero = pd.DataFrame({
    "sucursal": ["Bogota", "Medellin", "Cali"],
    "mes": ["Feb", "Feb", "Feb"],
    "ventas": [1200000, 2100000, 1600000]
})

marzo = pd.DataFrame({
    "sucursal": ["Bogota", "Medellin", "Cali"],
    "mes": ["Mar", "Mar", "Mar"],
    "ventas": [1500000, 2400000, 1800000]
})

# Consolidar las tablas
consolidado = pd.concat([enero, febrero, marzo], ignore_index=True)

# Convertir a formato ancho
tabla = consolidado.pivot_table(
    index="sucursal",
    columns="mes",
    values="ventas",
    aggfunc="sum"
)

# Calcular el porcentaje de crecimiento
tabla["crecimiento"] = (tabla["Mar"] - tabla["Ene"]) / tabla["Ene"] * 100

# Ordenar de mayor a menor
tabla = tabla.sort_values("crecimiento", ascending=False)

print(tabla)

# Mostrar la sucursal lider
print("Sucursal lider:", tabla.index[0])

mes           Ene      Feb      Mar  crecimiento
sucursal                                        
Bogota    1000000  1200000  1500000         50.0
Cali      1500000  1600000  1800000         20.0
Medellin  2000000  2100000  2400000         20.0
Sucursal lider: Bogota


In [46]:
import pandas as pd
import sqlite3

# data
pagos = pd.DataFrame({
    "pago.medio": ["Efectivo", "Tarjeta", "Transferencia",
                   "Efectivo", "Tarjeta", "Transferencia"],
    "ventas": [150000, 250000, 350000,
               210000, 350000, 600000]
})

# Solucion con Pandas
resultado_pandas = pagos.groupby("pago.medio")["ventas"].sum()
resultado_pandas = resultado_pandas.sort_values(ascending=False)

print("Resultado Pandas:")
print(resultado_pandas)

# conexion y tabla en SQLite
conexion = sqlite3.connect(":memory:")

pagos.to_sql("pagos", conexion, index=False, if_exists="replace")

# Solucion con SQL
consulta = """
SELECT "pago.medio", SUM(ventas) AS ventas
FROM pagos
GROUP BY "pago.medio"
ORDER BY ventas DESC
"""

resultado_sql = pd.read_sql_query(consulta, conexion)

print("\nResultado SQL:")
print(resultado_sql)

# Comparar los resultados
print("\nMedio lider Pandas:", resultado_pandas.index[0])
print("Medio lider SQL:", resultado_sql["pago.medio"][0])

print("Son iguales:", resultado_pandas.reset_index().equals(resultado_sql))

conexion.close()

Resultado Pandas:
pago.medio
Transferencia    950000
Tarjeta          600000
Efectivo         360000
Name: ventas, dtype: int64

Resultado SQL:
      pago.medio  ventas
0  Transferencia  950000
1        Tarjeta  600000
2       Efectivo  360000

Medio lider Pandas: Transferencia
Medio lider SQL: Transferencia
Son iguales: True
